In [1]:
import papermill as pm
import numpy as np
# Optuna
#!pip install optuna
import optuna

# GRID SEACH, define a parameter space and evaluate the simulation at each point uniformly

In [2]:
temperature        = [4e-3, 5e-3, 6e-3, 8e-3]   # mK
temperature_transv = [4e-3, 5e-3, 6e-3, 8e-3]   # mK
tau_mixing         = [15, 20, 25, 30, 35] # s
theta              = [10*np.pi/180, 15*np.pi/180, 20*np.pi/180, 25*np.pi/180] 
print(temperature, temperature_transv)


import multiprocessing as mp
import papermill as pm

def run_simulation(args):
    t1, t2, tau, angle = args

    output_name = f"Simulation_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    print(f"\n>>> Executing {output_name}")

    pm.execute_notebook(
        "Simulation.ipynb",
        f"output/notebooks/{output_notebook}_{bias}.ipynb",
        parameters={
            'Temperature'       : t1,
            'Temperature_transv': t2,
            'tau_mixing'        : tau,
            'theta'             : angle,
            'stringa'           : f"tau_{tau}s_theta_{int(angle*180/np.pi)}",
            'bias'              : "0g"
        }
    )


if __name__ == "__main__":
    # genera tutte le combinazioni (equivalente ai due for annidati)
    tasks = [(t1, t2, tau, angle) for t1 in temperature for t2 in temperature_transv for tau in tau_mixing for angle in theta]

    # numero di processi (non saturare la macchina)
    n_proc = min(len(tasks), max(1, mp.cpu_count() - 1))

    with mp.Pool(processes= 7, maxtasksperchild=1) as pool:
        pool.map(run_simulation, tasks)

[0.004, 0.005, 0.006, 0.008] [0.004, 0.005, 0.006, 0.008]

>>> Executing Simulation_4.0mK_4.0mK
>>> Executing Simulation_4.0mK_5.0mK
>>> Executing Simulation_4.0mK_8.0mK
>>> Executing Simulation_4.0mK_6.0mK
>>> Executing Simulation_4.0mK_4.0mK
>>> Executing Simulation_4.0mK_5.0mK
>>> Executing Simulation_4.0mK_8.0mK







>>> Executing Simulation_5.0mK_4.0mK

>>> Executing Simulation_5.0mK_4.0mK

>>> Executing Simulation_5.0mK_5.0mK

>>> Executing Simulation_5.0mK_6.0mK

>>> Executing Simulation_5.0mK_6.0mK

>>> Executing Simulation_5.0mK_8.0mK

>>> Executing Simulation_5.0mK_8.0mK

>>> Executing Simulation_6.0mK_4.0mK


>>> Executing Simulation_6.0mK_5.0mK
>>> Executing Simulation_6.0mK_5.0mK

>>> Executing Simulation_6.0mK_6.0mK

>>> Executing Simulation_6.0mK_6.0mK

>>> Executing Simulation_6.0mK_8.0mK

>>> Executing Simulation_8.0mK_4.0mK


>>> Executing Simulation_8.0mK_4.0mK
>>> Executing Simulation_8.0mK_5.0mK


>>> Executing Simulation_8.0mK_5.0mK
>>> Executing Simulation_8.0m

NameError: name 'output_notebook' is not defined

# Bayesian optimization, smart search of the minimum.

In [ ]:
def objective(trial):
    t1    = trial.suggest_float("Temperature", 0.5e-3, 10e-3)
    t2    = trial.suggest_float("Temperature_transv", 0.5e-3, 10e-3)
    tau   = trial.suggest_float("tau_mixing", 5, 100)
    angle = trial.suggest_float("theta", 5*np.pi/180, 180*np.pi/180)

    output_notebook = f"Simulation_tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    output_name     = f"tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_axial_{t1*1e3:.2f}mK_transv_{t2*1e3:.2f}mK"
    print(f"\n>>> Executing {output_name}")
    
    pm.execute_notebook(
        "Simulation.ipynb",
        f"output/{output_notebook}.ipynb",
        parameters={
            'Temperature'       : t1,
            'Temperature_transv': t2,
            'tau_mixing'        : tau,
            'theta'             : angle,
            'stringa'           : output_name,
            'bias'              : "0g"
        }
    )

    data = np.load("output/" + output_name)
    return float(data["metric"])

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=200)

In [ ]:
print("Best LR:", study.best_value)
print("Best params:", study.best_params)

# Simulation with comparison S-curve and Time Distributions
The objective function will perform a complete simulation, extracting simulated time distributions and comparing it to data time distributions. The objective function will also perform a simulation to extract the Scurve and compare it to the data. The metric will be the normalized LR of the time distributions 

In [15]:
list_biases = ['-0p75g', '0p0g', '0p5g', '0p75g', '-1p25g', '-0p37g', '0p25g', '1p25g', '-0p5g', '-0p25g']

def objective(trial):
    t1    = trial.suggest_float("Temperature", 0.5e-3, 20e-3)
    t2    = trial.suggest_float("Temperature_transv", 0.5e-3, 20e-3)
    tau   = trial.suggest_float("tau_mixing", 0, 100)
    angle = trial.suggest_float("theta", 0*np.pi/180, 180*np.pi/180)

    output_notebook = f"Simulation_tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    output_name     = f"tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_axial_{t1*1e3:.2f}mK_transv_{t2*1e3:.2f}mK"
    print(f"\n>>> Executing {output_name}")

    for bias in list_biases:
        pm.execute_notebook(
            "Simulation.ipynb",
            f"output/notebooks/{output_notebook}_{bias}.ipynb",
            parameters={
                'Temperature'       : t1,
                'Temperature_transv': t2,
                'tau_mixing'        : tau,
                'theta'             : angle,
                'stringa'           : output_name,
                'bias'              : bias,
                'nAtoms'            : 3000,
            }
        )

    pm.execute_notebook(
            "Metric_Worker.ipynb",
            f"output/notebooks/Metric_Worker.ipynb",
            parameters={
                'outputfile' : output_name
            }
        )

    
    data = np.load("output/" + output_name + "_0p0g.npz")  # take the LR from the 0g files output.

    LR = data["metric"]
    Chisq = data['Chisq_S']

    print(f"Time Annihilation: {LR:.4f}, Scurve {Chisq}" )
    
    return float( LR + np.mean(Chisq) )

In [16]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=1000)

[I 2026-02-08 09:56:22,438] A new study created in memory with name: no-name-58141724-c6e2-459c-8c85-e9acbf0cd076



>>> Executing tau_38.00s_theta_58_axial_15.46mK_transv_0.95mK


Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

[W 2026-02-08 14:15:22,877] Trial 13 failed with parameters: {'Temperature': 0.019303543122746555, 'Temperature_transv': 0.012625189763312197, 'tau_mixing': 83.30708398489463, 'theta': 0.5311180384949139} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/adriano/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_69193/1829027640.py", line 14, in objective
    pm.execute_notebook(
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/execute.py", line 116, in execute_notebook
    nb = papermill_engines.execute_notebook_with_engine(
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/engines.py", line 48, in execute_notebook_with_engine
    return self.get_engine(engine_name).execute_notebook(nb, kernel_name, **kwargs)
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/engines.py", line 370, in exe

KeyboardInterrupt: 